In [103]:
import tensorflow as tf
import numpy as np
import os

print(os.getcwd())
tfrecordpath = "../Data/tfrecords/"


/mnt/c/Users/alexs/Desktop/levbot/Training


### Load in data
#### Define schema


In [104]:
def decode(record_bytes):
    # Function for parsing each record in the tf files
    example = tf.io.parse_single_example(
        # Data
        record_bytes,

        # Schema
        {
        'Timeframe': tf.io.FixedLenFeature([], tf.string),
        'timestamp': tf.io.RaggedFeature(dtype=tf.int64),
        'Open': tf.io.RaggedFeature(dtype=tf.float32),
        'High': tf.io.RaggedFeature(dtype=tf.float32),
        'Low': tf.io.RaggedFeature(dtype=tf.float32),
        'Close': tf.io.RaggedFeature(dtype=tf.float32),
        'Volume': tf.io.RaggedFeature(dtype=tf.float32),
        }
        )

    return example

In [105]:
def getDataset(path):
    ds = tf.data.TFRecordDataset(path,  num_parallel_reads = tf.data.AUTOTUNE)
    ds = ds.map(decode, num_parallel_calls = tf.data.AUTOTUNE)
    return ds



In [106]:
import TensorSlider

In [127]:
windowsize = 100
lookahead = 5

def slider():


    path = tfrecordpath + "BTCUSD_PERP/{tframe}.tfrecord"
    datasets = {"1m" : getDataset(path.format(tframe="1m")),
                "5m":  getDataset(path.format(tframe="5m")),
                "15m":  getDataset(path.format(tframe="15m")),
                "30m":  getDataset(path.format(tframe="30m")),
                "1h":  getDataset(path.format(tframe="1h"))}
    return TensorSlider.WindowSlider(windowsize, lookahead, datasets)

dataset = tf.data.Dataset.from_generator(slider, 
    output_signature=(
        tf.TensorSpec((5,5,windowsize), dtype=tf.float32),
        tf.TensorSpec((lookahead+1,5), dtype=tf.float32))
    )

In [151]:
def createLabels(data, lookforward):
    """
    create input labels from the lookaehad data
    """
    
    ohlc = lookforward
    ohlc /= ohlc[0,:] # divide by open price
    ohlc -= 1 # zero out
    ohlc *= 100 # convert to 1/10 percentage
    
    high = ohlc[tf.math.argmin(ohlc[:,1]), 1]
    low = ohlc[tf.math.argmin(ohlc[:,2]), 2]

    # same process, compute relative deltas
    ohlc = data

    ohlc /= tf.expand_dims(ohlc[:,:,0],2)
    ohlc -= 1 # zero out
    ohlc *= 100 # convert to 1/10 percentage
    
    return tf.expand_dims(ohlc,0), [low, high]
    
    
    

In [152]:
import time
t0 = None
for i, data in enumerate(dataset.map(createLabels)):
    if i == 0:
        t0 = time.time()
    if i > 10:
        tdelta = time.time() - t0
        print(f"Time taken for {i+1} calls: {tdelta}s, thats {((tdelta/(i+1))*1000):.2f}ms per call")
        print(data[1])
        break
        

../Data/tfrecords/BTCUSD_PERP/1m.tfrecord
Time taken for 12 calls: 0.12799644470214844s, thats 10.67ms per call
tf.Tensor([-0.5598664 -0.5850792], shape=(2,), dtype=float32)


In [154]:
import keras
path = "models/modeltest/model.keras"
model = keras.models.load_model(path)
history = model.fit(dataset.map(createLabels),
                        epochs=1, verbose=1,
                        validation_data=None, callbacks=None)

../Data/tfrecords/BTCUSD_PERP/1m.tfrecord
    702/Unknown 14s 14ms/step - MeanAbsolutePercentageError: nan - MeanSquaredError: nan - loss: nan

KeyboardInterrupt: 